# fastText — Enriching Word Vectors with Subword Information (2016)

_Papers · Sequence & NLP_

**Build each word's vector by summing the vectors of its character n-grams, so even unseen and rare words get sensible embeddings.**

---

This notebook is a practice scaffold for the **fastText — Enriching Word Vectors with Subword Information (2016)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn.functional as F
torch.manual_seed(0)

# ---------- fastText's novel part: char n-gram bag + SUMMED word vector ----------
def ngrams(word, nmin=3, nmax=6):
    """Wrap in <...>, take all n-grams of length nmin..nmax, plus the whole word as a special token."""
    w = "<" + word + ">"
    gr = set()
    for n in range(nmin, nmax + 1):
        for i in range(len(w) - n + 1):
            gr.add(w[i:i+n])
    gr.add("<" + word + ">")          # the whole word, kept as one special n-gram
    return sorted(gr)

print("where 3-grams:", ngrams("where", 3, 3))   # ['<wh', '<where>', 'ere', 'her', 're>', 'whe']  (matches the paper)

# ---------- recompute the WORKED EXAMPLE (D=3, n=3) ----------
# named n-gram vectors so the sum is transparent
z = {"<ca":[0.2,0.5,-0.1], "cat":[0.4,-0.2,0.3], "at>":[-0.1,0.1,0.6], "<cat>":[0.3,0.2,-0.2],
     "ats":[0.05,-0.05,0.1], "ts>":[0.0,0.1,-0.05], "<cats>":[0.1,0.0,0.0]}
Z3 = {k: torch.tensor(v) for k, v in z.items()}
def sumvec(keys): return torch.stack([Z3[k] for k in keys]).sum(0)
v_cat  = sumvec(["<ca","cat","at>","<cat>"])               # "cat"
v_cats = sumvec(["<ca","cat","ats","ts>","<cats>"])        # OOV "cats" (shares <ca, cat)
cos = (v_cat @ v_cats) / (v_cat.norm() * v_cats.norm())
print("v(cat) :", [round(x,4) for x in v_cat.tolist()])    # [0.8, 0.6, 0.6]
print("v(cats):", [round(x,4) for x in v_cats.tolist()])   # [0.75, 0.35, 0.25]
print("cosine(cat, cats):", round(cos.item(), 4))          # 0.9521

# ---------- train a tiny fastText on a corpus that NEVER contains "cats" ----------
torch.manual_seed(1)
corpus = ("the cat sat the dog ran the cat ran the dog sat a cat plays a dog plays "
          "the cat eats the dog eats cat and dog cat or dog fish swims the fish swam "
          "fish and fish the fish eats").split()
vocab = sorted(set(corpus)); stoi = {w:i for i,w in enumerate(vocab)}; V = len(vocab)
D, K = 16, 4096                                            # K = n-gram hash buckets (paper uses 2e6)
def bucket_ids(word): return sorted({hash(g) % K for g in ngrams(word)})

Z    = (torch.randn(K, D) * 0.1).requires_grad_(True)      # n-gram (subword) input table
Vout = (torch.randn(V, D) * 0.1).requires_grad_(True)      # context (output) table
def wordvec(word): return Z[bucket_ids(word)].sum(0)       # <-- fastText: SUM of n-gram vectors

pairs = []                                                 # window = 2
for i, w in enumerate(corpus):
    for j in range(max(0,i-2), min(len(corpus),i+3)):
        if j != i: pairs.append((w, stoi[corpus[j]]))

opt = torch.optim.Adam([Z, Vout], lr=0.05)
g = torch.Generator().manual_seed(0)
for epoch in range(60):
    for cw, ctx in pairs:
        vc = wordvec(cw)                                   # summed center vector
        negs = torch.randint(0, V, (5,), generator=g)
        targets = torch.cat([torch.tensor([ctx]), negs])
        labels  = torch.tensor([1.,0.,0.,0.,0.,0.])
        s = Vout[targets] @ vc                             # scores for pos + 5 negatives
        loss = F.binary_cross_entropy_with_logits(s, labels)   # = negative-sampling logistic loss
        opt.zero_grad(); loss.backward(); opt.step()
print("final loss:", round(loss.item(), 4))

# ---------- OOV test: fastText vs word2vec ----------
def cosine(a, b): return (a @ b / (a.norm() * b.norm())).item()
with torch.no_grad():
    vc_cat, vc_dog, vc_fish = wordvec("cat"), wordvec("dog"), wordvec("fish")
    vc_cats = wordvec("cats")                              # OOV — fastText STILL returns a vector
print("\nfastText OOV 'cats' cosine to:")
print("  cat :", round(cosine(vc_cats, vc_cat), 3))       # highest — shares <ca, cat
print("  dog :", round(cosine(vc_cats, vc_dog), 3))
print("  fish:", round(cosine(vc_cats, vc_fish), 3))

# word2vec ABLATION: a whole-word lookup table has NO row for an unseen word
U = torch.randn(V, D)                                      # word2vec-style whole-word input table
def w2v(word): return U[stoi[word]]                        # raises KeyError for OOV
try:
    w2v("cats")
except KeyError:
    print("\nword2vec lookup('cats') -> KeyError: OOV has no vector at all.")

## Visualize the data & results

_Train a tiny fastText on a corpus that never contains 'cats'. Can it still build a vector for the out-of-vocabulary word 'cats' from its character n-grams, and does it land nearest 'cat' — where plain word2vec has no vector at all?_

In [ ]:
import torch, torch.nn.functional as F
torch.manual_seed(1)

def ngrams(word, nmin=3, nmax=6):
    w = "<" + word + ">"; gr = set()
    for n in range(nmin, nmax + 1):
        for i in range(len(w) - n + 1): gr.add(w[i:i+n])
    gr.add("<" + word + ">"); return sorted(gr)

corpus = ("the cat sat the dog ran the cat ran the dog sat a cat plays a dog plays "
          "the cat eats the dog eats cat and dog cat or dog fish swims the fish swam "
          "fish and fish the fish eats").split()
vocab = sorted(set(corpus)); stoi = {w:i for i,w in enumerate(vocab)}; V = len(vocab)
D, K = 16, 4096
def bucket_ids(word): return sorted({hash(g) % K for g in ngrams(word)})
Z    = (torch.randn(K, D) * 0.1).requires_grad_(True)
Vout = (torch.randn(V, D) * 0.1).requires_grad_(True)
def wordvec(word): return Z[bucket_ids(word)].sum(0)

pairs = []
for i, w in enumerate(corpus):
    for j in range(max(0,i-2), min(len(corpus),i+3)):
        if j != i: pairs.append((w, stoi[corpus[j]]))

opt = torch.optim.Adam([Z, Vout], lr=0.05)
g = torch.Generator().manual_seed(0)
for epoch in range(60):
    for cw, ctx in pairs:
        vc = wordvec(cw)
        targets = torch.cat([torch.tensor([stoi[corpus[0]] if False else ctx]),
                             torch.randint(0, V, (5,), generator=g)])
        labels = torch.tensor([1.,0.,0.,0.,0.,0.])
        loss = F.binary_cross_entropy_with_logits(Vout[targets] @ vc, labels)
        opt.zero_grad(); loss.backward(); opt.step()

def cosine(a, b): return round((a @ b / (a.norm() * b.norm())).item(), 3)
with torch.no_grad():
    cats = wordvec("cats")                       # OOV, never trained
    for w in ["cat", "fish", "dog"]:
        print(f"cosine(cats_OOV, {w:4s}) = {cosine(cats, wordvec(w))}")
# word2vec whole-word lookup would raise KeyError('cats') -> no OOV vector at all

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Build the character n-gram set $\mathcal{G}_w$ for the word "cat" using $n=3$ only (with boundary symbols and the whole-word token), then write its word vector as a sum.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Wrap the word: $\langle cat\rangle$. — _Boundary symbols let prefix/suffix differ from mid-word pieces._
- Take all length-3 windows: $\langle ca,\ cat,\ at\rangle$. — _Those are the 3-grams of the 5-character string <cat>._
- Add the whole word as a special token: $\langle cat\rangle$. — _The paper keeps the word itself in $\mathcal{G}_w$._
- Sum the vectors: $\mathbf{v}_{cat}=\mathbf{z}_{\langle ca}+\mathbf{z}_{cat}+\mathbf{z}_{at\rangle}+\mathbf{z}_{\langle cat\rangle}$. — _A word's vector is the sum of its n-gram vectors (boxed eq.)._

**Answer:** $\mathcal{G}_{cat}=\{\langle ca,\ cat,\ at\rangle,\ \langle cat\rangle\}$ and $\mathbf{v}_{cat}=\sum_{g\in\mathcal{G}_{cat}}\mathbf{z}_g$. With the lesson's numbers this is $[0.8,0.6,0.6]$.

</details>

**Problem 2.** OOV check: the word "cats" was never in training. Using $n=3$, which of its n-grams are SHARED with "cat", and why does that give it a sensible vector?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- n-grams of $\langle cats\rangle$: $\langle ca,\ cat,\ ats,\ ts\rangle$ plus $\langle cats\rangle$. — _All length-3 windows of the wrapped word, plus the whole-word token._
- Compare to $\mathcal{G}_{cat}=\{\langle ca, cat, at\rangle, \langle cat\rangle\}$. — _Find the overlap._
- Shared: $\langle ca$ and $cat$. — _Their vectors were already learned from "cat", and are reused by sharing._

**Answer:** "cats" shares $\langle ca$ and $cat$ with "cat", so its summed vector inherits most of "cat"'s representation and lands near it (cosine $\approx 0.95$ in the worked example) — with NO training on "cats". word2vec has no row for "cats" and cannot produce any vector.

</details>

**Problem 3.** Ablation: in the CODE, swap fastText's wordvec(word)=Z[ngrams(word)].sum(0) for a plain word2vec-style whole-word lookup U[stoi[word]]. What happens when you ask for the OOV word "cats", and what does this prove?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Replace the summed n-gram vector with a single whole-word row lookup. — _That is exactly the word2vec input representation._
- Call the model on "cats", which is not in stoi. — _"cats" was held out of the training vocabulary._
- Observe the KeyError (no row exists), then re-run the fastText version on "cats". — _Compare what each model can produce for an unseen word._

**Answer:** The word2vec lookup raises a KeyError — there is no vector for an unseen word, so OOV fails entirely. fastText still returns a vector for "cats" (sum of its n-grams, mostly shared with "cat") and it lands nearest "cat". This is the paper's core advantage: composing words from subword pieces makes OOV and rare words work, which whole-word embeddings cannot.

</details>